# Delisting notices, auditor changes and layoffs: red-flag 8-Ks in 2026 against 2025

Count five red-flag 8-K items for the first nine months of 2026 and 2025, chart delisting notices month by month, and find the companies that changed auditors and got a delisting notice in the same year.

Read it with charts at https://www.quantisnow.com/sec-api/notebooks/red-flag-8ks-2026. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

A few 8-K items almost never bring good news: a stock exchange saying the company no longer meets its listing rules (Item 3.01), a change of auditor (4.01), a restructuring with layoffs (2.05), past results that can't be relied on (4.02) and bankruptcy (1.03). This notebook counts them for January to September of 2026 and of 2025, then looks for companies that collected more than one.

The helper pages through a query 50 records at a time, one API call per page. Every notebook here starts with it.

In [1]:
import os, requests
import pandas as pd

API = "https://www.quantisnow.com/api/v1"
KEY = os.environ["QUANTISNOW_API_KEY"]

def search(dataset, query, start=0, size=50, sort=None):
    body = {"query": query, "from": start, "size": size}
    if sort:
        body["sort"] = sort
    r = requests.post(f"{API}/{dataset}", json=body,
                      headers={"Authorization": f"Bearer {KEY}"})
    r.raise_for_status()
    return r.json()

def search_all(dataset, query, sort=None):
    """Every record that matches, 50 per call."""
    rows, start = [], 0
    while True:
        page = search(dataset, query, start, 50, sort)
        rows += page["data"]
        start += 50
        if start >= min(page["total"]["value"], 10_000):
            return rows

## Count without downloading

Every response carries `total`, the number of matching records. Ask for one record and read the total, and each count costs one call.

In [2]:
ITEMS = {"3.01": "delisting notice", "4.01": "auditor change", "2.05": "layoffs",
         "4.02": "non-reliance", "1.03": "bankruptcy"}

def count(query):
    return search("8-k/events", query, size=1)["total"]["value"]

rows = []
for item, what in ITEMS.items():
    a = count(f'items.item:"{item}" AND filedAt:[2025-01-01 TO 2025-09-30]')
    b = count(f'items.item:"{item}" AND filedAt:[2026-01-01 TO 2026-09-30]')
    rows.append({"item": item, "what": what, "Jan-Sep 2025": a, "Jan-Sep 2026": b,
                 "change": f"{(b - a) / a:+.0%}"})
pd.DataFrame(rows)

item,what,Jan-Sep 2025,Jan-Sep 2026,change
3.01,delisting notice,"1,095",863,-21%
4.01,auditor change,418,260,-38%
2.05,layoffs,175,137,-22%
4.02,non-reliance,85,62,-27%
1.03,bankruptcy,26,31,+19%


Four of the five fell. Auditor changes dropped the most, 38%, from 418 to 260. Bankruptcy filings are the exception, up from 26 to 31. The numbers are small, but they rose while the other four fell.

## Delisting notices month by month

In [3]:
import calendar

monthly = {}
for year in (2025, 2026):
    monthly[year] = []
    for m in range(1, 10):
        last = calendar.monthrange(year, m)[1]
        monthly[year].append(count(f'items.item:"3.01" AND filedAt:[{year}-{m:02d}-01 TO {year}-{m:02d}-{last}]'))
pd.DataFrame(monthly, index=[calendar.month_abbr[m] for m in range(1, 10)]).plot.bar(
    title="Item 3.01 delisting notices by month")

Item 3.01 delisting notices by month
Jan	142	94
Feb	105	76
Mar	110	90
Apr	196	123
May	156	111
Jun	89	57
Jul	103	113
Aug	111	112
Sep	83	87

Delisting notices ran below 2025 for the first half of the year, then caught up: July, August and September of 2026 came in at or above the same months of 2025.

## Companies with two red flags

An auditor walking away and an exchange warning about the listing, in the same year, is a combination worth a closer look. Pull both lists for 2026 and join them on the company's CIK. Then mark the ones that filed for bankruptcy later in the year.

In [4]:
W = "filedAt:[2026-01-01 TO 2026-09-30]"
def first_by_company(query, window=W):
    df = pd.json_normalize(search_all("8-k/events", f"{query} AND {window}", sort="filedAt:asc"))
    return df.drop_duplicates("cik").set_index("cik")

auditor = first_by_company('items.item:"4.01"')
delist = first_by_company('items.item:"3.01"')
bankrupt = first_by_company('items.item:"1.03"', "filedAt:[2026-01-01 TO 2026-11-21]")
both = auditor.index.intersection(delist.index)
print(f"auditor changes: {len(auditor)} companies")
print(f"delisting notices: {len(delist)} companies")
print(f"both: {len(both)} companies")

auditor changes: 230 companies
delisting notices: 584 companies
both: 60 companies

In [5]:
pd.DataFrame({
    "ticker": auditor.loc[both, "ticker"],
    "companyName": auditor.loc[both, "companyName"],
    "auditor_change": auditor.loc[both, "filedAt"].str[:10],
    "delisting_notice": delist.loc[both, "filedAt"].str[:10],
    "bankruptcy": bankrupt["filedAt"].str[:10].reindex(both),
}).sort_values("auditor_change").head(15)

ticker,companyName,auditor_change,delisting_notice,bankruptcy
NUTR,Nusatrip Incorporated,2026-01-06,2026-08-07,
LASE,LASE Corp.,2026-01-06,2026-04-21,
SGBX,Safe & Green Holdings Corp.,2026-01-09,2026-08-06,2026-05-04
VRAR,The Glimpse Group Inc.,2026-01-09,2026-03-18,
NUKK,Nukkleus Inc.,2026-01-14,2026-05-08,
BRLS,Borealis Foods Inc.,2026-01-16,2026-01-16,
DAIC,CID HoldCo Inc.,2026-01-20,2026-02-11,
STAI,ScanTech AI Systems Inc.,2026-01-21,2026-02-09,
NKLR,Terra Innovatum Global N.V.,2026-01-21,2026-04-17,
NCRA,Nocera Inc.,2026-02-03,2026-02-03,


Sixty companies did both. Most are small caps where a listing problem and an auditor change often have one cause: the company can't pay for, or pass, an audit. Safe & Green Holdings changed auditors in January and filed for bankruptcy in May, three months before its delisting notice.

## Take it further

Add a third flag by joining a full-text search for "going concern" in 10-Qs; the [full-text guide](https://www.quantisnow.com/sec-api/full-text-search) shows the query. To watch these as they happen, run the same queries on a schedule with `filedAt:>=` set to yesterday.